In [0]:
# ==================================================
# SILVER — Limpeza e Padronização
# Pipeline PRF Acidentes de Trânsito
# ==================================================

from pyspark.sql.functions import (
    col, to_timestamp, to_date, month, year, 
    dayofweek, upper, trim, regexp_replace,
    when, round, current_timestamp, lit
)

# Lê da Bronze
df = spark.table("workspace.default.bronze_acidentes")

print(f"Registros lidos da Bronze: {df.count()}")
print(f"Schema:")
df.printSchema()

In [0]:
# ==================================================
# Transformações Silver
# ==================================================

df_silver = (df
    # 1. Cast de tipos
    .withColumn("data", to_date(col("data_inversa"), "yyyy-MM-dd"))
    .withColumn("horario", to_timestamp(col("horario"), "HH:mm:ss"))
    .withColumn("km", regexp_replace(col("km"), ",", ".").cast("double"))
    .withColumn("latitude", regexp_replace(col("latitude"), ",", ".").cast("double"))
    .withColumn("longitude", regexp_replace(col("longitude"), ",", ".").cast("double"))

    # 2. Extração temporal
    .withColumn("ano", year(col("data")))
    .withColumn("mes", month(col("data")))
    .withColumn("dia_semana_num", dayofweek(col("data")))

    # 3. Padronização de texto
    .withColumn("causa_acidente", trim(upper(col("causa_acidente"))))
    .withColumn("tipo_acidente", trim(upper(col("tipo_acidente"))))
    .withColumn("uf", trim(upper(col("uf"))))
    .withColumn("municipio", trim(upper(col("municipio"))))
    .withColumn("fase_dia", trim(upper(col("fase_dia"))))
    .withColumn("condicao_metereologica", trim(upper(col("condicao_metereologica"))))

    # 4. Trata nulos nas colunas críticas
    .withColumn("classificacao_acidente", 
        when(col("classificacao_acidente").isNull(), "NAO INFORMADO")
        .otherwise(trim(upper(col("classificacao_acidente"))))
    )

    # 5. Remove colunas desnecessárias pra Silver
    .drop("data_inversa", "regional", "delegacia", "uop", "ingested_at", "ano_particao")

    # 6. Timestamp de processamento
    .withColumn("processed_at", current_timestamp())
)

print(f"Registros Silver: {df_silver.count()}")
df_silver.printSchema()

In [0]:
# ==================================================
# Grava Silver no Delta Lake
# ==================================================

SILVER_TABLE = "workspace.default.silver_acidentes"

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("ano", "mes")
    .saveAsTable(SILVER_TABLE)
)

print("Silver gravada com sucesso!")
print(f"Total de registros: {df_silver.count()}")

In [0]:
# Validação Silver
spark.sql("""
    SELECT 
        ano, mes, 
        COUNT(*) as total_acidentes,
        SUM(mortos) as total_mortos,
        SUM(feridos) as total_feridos
    FROM workspace.default.silver_acidentes
    GROUP BY ano, mes
    ORDER BY ano, mes
""").show()